In [1]:
import pandas as pd
import numpy as np

# Methodology

Sentiment labels are assigned to each news article based on the sign of this aggregated `three-day excess return`. This excess return is calculated from the day a news article is first published and extends over the two subsequent days. To elaborate, excess return is defined as the difference between the return of a particular stock and the overall market return on the same day. This calculation is not limited to the day the news is published; instead, it aggregates the returns for the following two days as well, providing a
comprehensive three-day outlook.

A positive aggregated excess return leads to a sentiment label of `1`, indicating a positive sentiment. Conversely, a non-positive aggregated excess return results in a sentiment label of `0`, suggesting a negative sentiment.

In [2]:
stocks = pd.read_excel("./Data/stocks_data.xlsx", header=[0, 1], index_col=0)
stocks_info = pd.read_excel("./Data/SPY_companies_info.xlsx")


In [3]:
# Separate DataFrames by the top-level column (first row of headers)
stocks_dict = {key: stocks[key] for key in stocks.columns.levels[0]}


In [4]:
market = pd.read_excel("./Data/market_data.xlsx")
market = market.set_index('Date')

In [5]:
def excess_return_sentiment_label(stock_data, market_data, num_days=3, price_used="Adj Close"):
    # Make a copy so as not to modify original
    stock_df = stock_data.copy()
    market_df = market_data.copy()


    # Filter for necessary price columns
    stock_df = stock_df[[price_used]].copy()
    market_df = market_df[[price_used]].copy()
    market_df = market_df.add_prefix("SPY_")

    # Calculate daily returns
    merged_df = stock_df.join(market_df)
    merged_df = merged_df.pct_change()

    # Calculate specified X-days aggregated returns
    merged_df["excess_returns"] = merged_df["Adj Close"] - merged_df["SPY_Adj Close"]
    # merged_df["X_days_excess_returns"] =\
    #     (
    #         merged_df["excess_returns"]
    #         .rolling(num_days)
    #         .sum()
    #         .shift(-(num_days - 1))
    #     )
    
    merged_df["X_days_excess_returns"] =\
        (
            (1 + merged_df["excess_returns"])
            .rolling(num_days)
            .apply(lambda x: x.cumprod()[-1], raw=True)
            .shift(-(num_days - 1)) 
            - 1
        )
    # Create sentiment label
    merged_df["sentiment_label"] = np.sign(merged_df["X_days_excess_returns"])

    return merged_df

In [8]:
excess_return_sentiment_label(stocks_dict['AAPL'], market).head(10)

,Adj Close,SPY_Adj Close,excess_returns,X_days_excess_returns,sentiment_label
Date,,,,,
2014-01-02,NaN,NaN,NaN,NaN,NaN
2014-01-03,-0.021966,-0.000164,-0.021802,-0.026745,-1.0
2014-01-06,0.005453,-0.002898,0.008351,0.001031,1.0
2014-01-07,-0.007152,0.006142,-0.013294,-0.020587,-1.0
2014-01-08,0.006333,0.000218,0.006115,-0.016717,-1.0
2014-01-09,-0.012770,0.000654,-0.013424,-0.004573,-1.0
2014-01-10,-0.006673,0.002722,-0.009395,0.018052,1.0
2014-01-13,0.005235,-0.013305,0.018540,0.042802,1.0
2014-01-14,0.019898,0.010897,0.009001,0.019437,1.0


In [10]:
news = pd.ExcelFile('./Data/news_data_final_filtered_train1.xlsx')
news_dict = {
    sheet_name: news.parse(sheet_name) for sheet_name in news.sheet_names
}


In [38]:
def label_all_news(news_dictionary):
    labelled_news_dict = {}

    for k, v in news_dict.items():

        try:
            # Prepare the excess_returns dataframe
            excess_returns_df = excess_return_sentiment_label(
                stocks_dict[k], market
            )

            # Set date column of news dataframe to datetime
            temp_df = v.copy()
            temp_df["date"] = pd.to_datetime(temp_df["date"]).dt.normalize()
            temp_df["sentiment_label"] = temp_df["date"].apply(
                lambda x: (
                    excess_returns_df.loc[x]["sentiment_label"]
                    if x in excess_returns_df.index
                    else np.nan
                )
            )
            temp_df = temp_df.dropna()
            
            labelled_news_dict[k] = temp_df
            
        except:
            pass
    
    return labelled_news_dict

In [39]:
train1_labelled = pd.concat(
    [df.assign(Ticker=ticker) for ticker, df in label_all_news(news_dict).items()],
    ignore_index=True,
)

In [41]:
news = pd.ExcelFile('./Data/news_data_final_filtered_train2.xlsx')
news_dict = {
    sheet_name: news.parse(sheet_name) for sheet_name in news.sheet_names
}

In [42]:
train2_labelled = pd.concat(
    [df.assign(Ticker=ticker) for ticker, df in label_all_news(news_dict).items()],
    ignore_index=True,
)

In [46]:
train_df = pd.concat([train1_labelled, train2_labelled], ignore_index=True)


In [48]:
news2 = pd.ExcelFile('./Data/news_data_final_filtered_test1.xlsx')
news_dict2 = {
    sheet_name: news2.parse(sheet_name) for sheet_name in news2.sheet_names
}

In [49]:
test1_labelled = pd.concat(
    [df.assign(Ticker=ticker) for ticker, df in label_all_news(news_dict2).items()],
    ignore_index=True,
)

In [50]:
news2 = pd.ExcelFile('./Data/news_data_final_filtered_test2.xlsx')
news_dict2 = {
    sheet_name: news2.parse(sheet_name) for sheet_name in news2.sheet_names
}

In [51]:
test2_labelled = pd.concat(
    [df.assign(Ticker=ticker) for ticker, df in label_all_news(news_dict2).items()],
    ignore_index=True,
)

In [52]:
test_df = pd.concat([test1_labelled, test2_labelled], ignore_index=True)


In [53]:
train_df.to_csv("./Data/news_data_final_train_labelled.csv", index=False)

In [54]:
test_df.to_csv("./Data/news_data_final_test_labelled.csv", index=False)